# Agentic_RL_Exec: GSM8K GRPO tool arm (Qwen2.5-0.5B, 100 step, SandboxTool)

**할 일(사람)**: 런타임을 A100으로 설정 → Run all → **1분 내** Drive 인증 팝업에서 "Allow" 클릭.

**중간 개입 필요(1회)**: 학습 셀이 step5 근처에서 `timing_s/step`을 출력하면 로그를 보고
100 step 전체 예상 소요시간을 계산해서 알려줌 — tool arm은 코드실행 왕복이 추가돼
control arm(≈38분)보다 오래 걸릴 것으로 예상(미검증 추정). 같은 체크포인트에서
`num_turns/max`도 확인 — step1~5 전부 2.0이면 tool이 한 번도 안 불렸다는 뜻(조기중단
경고 출력됨, GRPO는 안 일어나는 행동을 강화 못 하므로 이 경우 결과 자체가 발견임).

이 노트북은 브라우저에서 사람이 Run all로 직접 돌리는 걸 전제로 함(colab-cli 세션 소실
버그 #106 회피 — 상세: vault Colab_Ops-MOC). Drive에 결과 로그를 저장해 세션이 죽어도
산출물이 남게 함.

**목적**: control arm(`gsm8k_grpo_control_100step.ipynb`, 2026-09-27 완주 성공,
step0=0.08%→step100=48.0%)과 tool 유무만 다르게 비교. agent_loop=`tool_agent`
동일, `SandboxTool`(코드실행기)만 tools에 추가. 코드가 계산 결과를 반환하고 모델이
그걸 `#### N`으로 출력하는 것도 tool-use의 정당한 경로로 간주(reward 설계 결정,
2026-09-27) — 단, 정답이 tool 출력에서 왔는지 모델 자체 텍스트에서 왔는지는
셀13에서 attribution 분리 집계.

**SandboxTool 설계(원본 tutorial과 다름, advisor 검토 반영, 2026-09-27)**:
- FastAPI+Ray actor 서버 방식 제거 — 학습은 별도 `bash` subprocess로 뜨는데,
  이 노트북 커널의 Ray/Python 환경(시스템 Python)과 학습 프로세스의 uv venv가
  달라 커널에서 띄운 서버를 학습쪽이 못 씀. 대신 `SandboxTool.execute`가
  코드를 직접 subprocess 실행(원본 tutorial의 서버도 결국 bare subprocess였음,
  격리 수준은 동일).
- timeout 30s→10s로 축소(GSM8K 산술은 수 ms면 끝남), `asyncio.wait_for`+`kill()`.
- 동시 실행 제한(`asyncio.Semaphore`) — 100개 rollout이 동시에 코드 실행하면
  vLLM과 CPU 자원을 두고 경쟁.
- 실행 격리: uid/gid 65534(nobody)로 subprocess 실행 — 모델이 생성한 코드가
  root 권한으로 Drive(읽기전용 아님, rw 마운트) 접근 못 하게. stdlib
  `python3 -c`만 허용(uv venv는 root 소유라 65534가 실행 못 함).
- tool 클래스는 `/content/verl/verl/tools/gsm8k_sandbox_tool.py`(verl 패키지
  내부)에 둠 — `class_name: verl.tools.gsm8k_sandbox_tool.SandboxTool`로
  `importlib.import_module` 표준 경로 resolve(패키지 밖에 두면 학습
  subprocess의 sys.path에 없어 import 실패 위험, advisor 지적).

**정규식 버그 정정**(control arm 실행 후 발견, 여기 반영됨): 실제 val 로그값은
`val-core/openai/gsm8k/acc/mean@1:np.float64(0.000758...)` 형태로 `np.float64()`
래퍼가 씌워짐(control arm 노트북은 아직 미수정).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RUN_DIR = '/content/drive/MyDrive/agentic_rl/runs/' + __import__('datetime').datetime.now().strftime('%Y%m%d_%H%M%S')
os.makedirs(RUN_DIR, exist_ok=True)
print('RUN_DIR =', RUN_DIR)


## Setup: clone + env (idempotent)

In [ ]:
import os, subprocess

if not os.path.exists('/content/verl'):
    subprocess.run(['git', 'clone', '--depth', '1',
                     'https://github.com/verl-project/verl.git', '/content/verl'], check=True)
else:
    print('/content/verl already exists, skip clone')

print(subprocess.run(['git', '-C', '/content/verl', 'log', '--oneline', '-1'],
                      capture_output=True, text=True).stdout)


In [ ]:
import subprocess

# idempotent: skip uv sync if venv already has verl+vllm importable
check = subprocess.run(
    ['bash', '-c',
     'cd /content/verl && uv run --frozen --all-packages --extra vllm --extra fsdp '
     'python3 -c "import verl, vllm, torch; print(\'OK\', torch.cuda.is_available())" 2>&1'],
    capture_output=True, text=True
)
print(check.stdout[-1000:])
need_sync = 'OK True' not in check.stdout

if need_sync:
    print('running uv sync (several GB, may take a while)...')
    r = subprocess.run(['bash', '-c',
        'cd /content/verl && uv sync --frozen --all-packages --extra vllm --extra fsdp'],
        capture_output=True, text=True)
    print(r.stdout[-3000:])
    print('STDERR:', r.stderr[-2000:])
    print('exit', r.returncode)
else:
    print('venv already OK, skip uv sync')


In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout)
r2 = subprocess.run(['bash', '-c',
    'cd /content/verl && uv run --frozen --all-packages --extra vllm --extra fsdp python3 -c '
    '"import torch; print(torch.__version__, torch.version.cuda, torch.cuda.is_available(), torch.cuda.get_device_name(0))"'],
    capture_output=True, text=True)
print(r2.stdout)
print(r2.stderr[-500:])
assert 'True' in r2.stdout, 'CUDA not available, stop here'


## GSM8K preprocessing

In [ ]:
import subprocess, os
if not os.path.exists('/content/data/gsm8k/train.parquet'):
    r = subprocess.run(['bash', '-c',
        'cd /content/verl && uv run --frozen --all-packages --extra vllm --extra fsdp python3 '
        'examples/data_preprocess/gsm8k.py --local_save_dir /content/data/gsm8k'],
        capture_output=True, text=True)
    print(r.stdout[-2000:])
    print(r.stderr[-1000:])
else:
    print('gsm8k parquet already exists, skip')

r = subprocess.run(['ls', '-la', '/content/data/gsm8k/'], capture_output=True, text=True)
print(r.stdout)


## Code sandbox (SandboxTool) — verl 패키지 안에 설치, subprocess 직접실행+timeout+격리

셀0 설계 노트 참고. 다음 셀이 `verl.tools.gsm8k_sandbox_tool` 모듈 파일을
`/content/verl/verl/tools/`에 쓰고, `tool_config.json`을 만듦(학습 셀에서 참조).


In [ ]:
import json, os

# tool_config.json이 참조할 tool 클래스는 verl 패키지 내부에 둬야 학습
# subprocess(uv venv, editable-installed verl)의 sys.path에서 항상 resolve됨
# (verl/tools/tool_registry.py가 importlib.import_module(module_name)로 로드 —
# 표준 절대 import 규칙, 패키지 밖 경로는 sys.path 불확실).
sandbox_tool_module = '''
"""GSM8K tool arm: code sandbox tool. subprocess 직접실행(서버 없음),
10s timeout, 동시성 제한, uid 65534 격리. 원본은 verl tutorial
agent_loop_get_started의 SandboxTool(서버+Ray actor 방식) — 이 노트북 학습이
별도 subprocess로 뜨는 구조상 커널 Ray를 공유 못 해 서버 없이 직접실행으로
변경(2026-09-27, advisor 검토).
"""
import asyncio
import os
import re
import pwd

from transformers.utils import get_json_schema
from verl.tools.base_tool import BaseTool, OpenAIFunctionToolSchema, ToolResponse

_CODE_TIMEOUT_SEC = 10
_MAX_CONCURRENT = 8
_semaphore = asyncio.Semaphore(_MAX_CONCURRENT)

try:
    _NOBODY_UID = pwd.getpwnam("nobody").pw_uid
    _NOBODY_GID = pwd.getpwnam("nogroup").pw_gid
except KeyError:
    _NOBODY_UID = 65534
    _NOBODY_GID = 65534


def _drop_privileges():
    os.setgid(_NOBODY_GID)
    os.setuid(_NOBODY_UID)


class SandboxTool(BaseTool):
    def __init__(self, config: dict, tool_schema: OpenAIFunctionToolSchema):
        super().__init__(config, tool_schema)
        self.code_pattern = re.compile(r"```py(.*?)```", re.DOTALL)

    async def code_interpreter(self, code: str) -> str:
        """Execute the code in the sandbox.

        Args:
            code: The code to be executed.

        Returns:
            str: The output of the code execution.
        """
        async with _semaphore:
            try:
                process = await asyncio.create_subprocess_exec(
                    "/usr/bin/python3", "-c", code,
                    stdout=asyncio.subprocess.PIPE,
                    stderr=asyncio.subprocess.PIPE,
                    preexec_fn=_drop_privileges,
                    cwd="/tmp",
                )
            except Exception as e:
                return f"Error launching sandbox process: {e}"

            try:
                stdout, stderr = await asyncio.wait_for(
                    process.communicate(), timeout=_CODE_TIMEOUT_SEC
                )
                return stdout.decode() + stderr.decode()
            except asyncio.TimeoutError:
                process.kill()
                await process.wait()
                return f"(timed out after {_CODE_TIMEOUT_SEC}s, process killed)"

    def get_openai_tool_schema(self) -> OpenAIFunctionToolSchema:
        schema = get_json_schema(self.code_interpreter)
        return OpenAIFunctionToolSchema(**schema)

    async def execute(self, instance_id: str, parameters: dict, **kwargs) -> tuple[ToolResponse, float, dict]:
        code = parameters["code"]
        matches = self.code_pattern.findall(code)
        if matches:
            code = matches[0].strip()

        lines = code.split("\\n")
        for i, line in reversed(list(enumerate(lines))):
            if line == "":
                continue
            if not lines[i].startswith("print"):
                lines[i] = f"print({line})"
            break
        code = "\\n".join(lines)

        result = await self.code_interpreter(code)
        return ToolResponse(text=result), 0.0, {}
'''

os.makedirs("/content/verl/verl/tools", exist_ok=True)
with open("/content/verl/verl/tools/gsm8k_sandbox_tool.py", "w") as f:
    f.write(sandbox_tool_module)

tool_config = {
    "tools": [
        {
            "class_name": "verl.tools.gsm8k_sandbox_tool.SandboxTool",
            "config": {"type": "native"},
        },
    ],
}
tool_config_path = "/content/tool_config.json"
with open(tool_config_path, "w") as f:
    json.dump(tool_config, f)

print("wrote /content/verl/verl/tools/gsm8k_sandbox_tool.py and", tool_config_path)


## SandboxTool self-test (학습 전 필수, uv venv로 실행)

advisor 지적: import·timeout·권한 버그는 100step 유료 학습 도중 발견하면
비쌈 — 학습 시작 전 이 셀에서 3가지를 확인. 실패하면 학습 셀로 넘어가지 말 것.
1. 정상 코드(`print(2+3)`) → `5` 반환
2. 무한루프(`while True: pass`) → 약 10초 후 timeout 텍스트 반환(멈추지 않아야 함)
3. Drive 쓰기 시도 → PermissionError로 거부(uid 65534라 root 소유 Drive mount에 못 씀)


In [ ]:
import subprocess

_selftest_script = r'''
import asyncio, importlib, sys
sys.path.insert(0, "/content/verl")
mod = importlib.import_module("verl.tools.gsm8k_sandbox_tool")

async def main():
    tool = mod.SandboxTool(config={"type": "native"}, tool_schema=None)

    r1, _, _ = await tool.execute("t1", {"code": "2+3"})
    print("TEST1(정상 실행) result:", repr(r1.text))
    assert "5" in r1.text, f"expected 5 in output, got {r1.text!r}"

    r2, _, _ = await tool.execute("t2", {"code": "```py\nwhile True:\n    pass\n```"})
    print("TEST2(timeout) result:", repr(r2.text))
    assert "timed out" in r2.text, f"expected timeout text, got {r2.text!r}"

    r3, _, _ = await tool.execute(
        "t3",
        {"code": "open('/content/drive/MyDrive/agentic_rl/pwned.txt', 'w').write('x')"},
    )
    print("TEST3(Drive 격리) result:", repr(r3.text))
    assert "Permission" in r3.text or "Errno 13" in r3.text, (
        f"expected PermissionError, got {r3.text!r} -- Drive 격리 실패, 학습 중단할 것"
    )

    print("ALL SELFTESTS PASSED")

asyncio.run(main())
'''

r = subprocess.run(
    ['bash', '-c',
     f'cd /content/verl && uv run --frozen --all-packages --extra vllm --extra fsdp '
     f'python3 -c {subprocess.list2cmdline([_selftest_script])}'],
    capture_output=True, text=True,
)
print(r.stdout)
print("STDERR:", r.stderr[-1500:])
assert 'ALL SELFTESTS PASSED' in r.stdout, 'SandboxTool self-test 실패 — 학습 셀로 넘어가지 말 것'


## Training: GRPO, Qwen2.5-0.5B-Instruct, single GPU, 100 steps, tool_agent(SandboxTool)

control arm과 agent_loop 종류는 동일(`tool_agent`), 차이는 딱 하나:
`actor_rollout_ref.rollout.multi_turn.tool_config_path`가 위에서 만든
`tool_config.json`을 가리킴. `multi_turn.max_assistant_turns=3`으로 상한
설정(advisor 지적 — 무제한이면 코드 재시도 반복으로 한 스텝이 timeout
여러 번만큼 늘어질 수 있음).

**판정기준(control arm과 동일, falsifiable)**: `test_freq=10`으로 step10~100
strict val 측정, 마지막 3개 포인트(step80/90/100)가 각각 step0 대비 +0.05
이상이고 비감소면 "학습 진행"으로 판정.

**조기중단 조건(신규, advisor 지적)**: step1~5에서 `num_turns/max`가 전부
2.0이면 tool이 한 번도 호출 안 됐다는 뜻 — GRPO는 발생하지 않는 행동을
강화 못 하므로 나머지 95step은 control arm 재확인일 뿐. 이 경우 경고
출력하고 계속할지는 판단 필요(자동 중단은 안 함, Run all 전제라).


In [ ]:
import subprocess, os, time, re

env = os.environ.copy()
env.update({
    'MODEL_PATH': 'Qwen/Qwen2.5-0.5B-Instruct',
    'TRAIN_FILE': '/content/data/gsm8k/train.parquet',
    'TEST_FILE': '/content/data/gsm8k/test.parquet',
    'NGPUS_PER_NODE': '1',
    'ROLLOUT_TP': '1',
    'ROLLOUT_GPU_MEM_UTIL': '0.5',
    'ROLLOUT_N': '5',
    'TRAIN_BATCH_SIZE': '20',
    'PPO_MINI_BATCH_SIZE': '10',
    'PPO_MICRO_BATCH_SIZE_PER_GPU': '2',
    'LOG_PROB_MICRO_BATCH_SIZE_PER_GPU': '2',
    'MAX_PROMPT_LENGTH': '512',
    'MAX_RESPONSE_LENGTH': '1024',
    'ACTOR_LR': '1e-6',
    'PROJECT_NAME': 'agentic_rl_exec',
    'EXPERIMENT_NAME': 'gsm8k_grpo_qwen25_0_5b_tool_100step',
    'SAVE_FREQ': '-1',
    'TEST_FREQ': '10',
    'TOTAL_EPOCHS': '1',
})

val_dump_dir = os.path.join(RUN_DIR, 'val_dump')
log_path = os.path.join(RUN_DIR, 'train.log')
cmd = (
    'set -o pipefail; cd /content/verl && bash examples/grpo_trainer/run_qwen3_4b_fsdp.sh '
    'trainer.logger=\'["console"]\' '
    'trainer.total_training_steps=100 '
    'trainer.val_before_train=True '
    f'trainer.validation_data_dir={val_dump_dir} '
    'actor_rollout_ref.rollout.mode=async '
    'actor_rollout_ref.rollout.agent.default_agent_loop=tool_agent '
    f'actor_rollout_ref.rollout.multi_turn.tool_config_path={tool_config_path} '
    'actor_rollout_ref.rollout.multi_turn.max_assistant_turns=3 '
    'data.return_raw_chat=True '
    f'2>&1 | tee {log_path}'
)
print('$', cmd)

proc = subprocess.Popen(['bash', '-c', cmd], env=env)

step_pat = re.compile(r'step:(\d+)')
timing_pat = re.compile(r'timing_s/step:([0-9.]+)')
num_turns_max_pat = re.compile(r'(?<!val-aux/)num_turns/max:np\.\w+\(([0-9.]+)\)')

step5_reported = False
while proc.poll() is None:
    time.sleep(15)
    if not os.path.exists(log_path):
        continue
    with open(log_path) as f:
        log_tail = f.read()
    steps_seen = [int(m) for m in step_pat.findall(log_tail)]
    if not step5_reported and steps_seen and max(steps_seen) >= 5:
        timings = timing_pat.findall(log_tail)
        if timings:
            sample = timings[1:6] if len(timings) > 5 else timings
            avg = sum(float(t) for t in sample) / len(sample)
            print(f'[step5 체크포인트] step2~6 평균 timing_s/step={avg:.1f}s (하한 추정), '
                  f'100step 학습만 ≈ {avg*100/60:.1f}분 + val 11회 별도 소요 (계속 진행 중)')
        else:
            print('[step5 체크포인트] timing_s/step 로그 패턴 미발견(포맷 미검증) — 계속 진행')

        num_turns = num_turns_max_pat.findall(log_tail)
        if num_turns and all(float(v) == 2.0 for v in num_turns):
            print('[경고] step1~5의 train num_turns/max가 전부 2.0 — tool이 한 번도 '
                  '호출 안 됨. 나머지 95step은 control arm 재확인일 뿐일 수 있음. '
                  '계속 두면 자동 진행, 중단하려면 셀 실행 중지.')
        elif num_turns:
            print(f'[확인] tool 호출 발생(num_turns/max 샘플: {num_turns[:5]})')
        else:
            print('[경고] num_turns/max 로그 패턴 미발견(포맷 미검증)')
        step5_reported = True
    if step5_reported:
        proc.wait()
        break

if proc.poll() is None:
    proc.wait()
print('exit', proc.returncode, '(pipefail 적용 — 0이 아니면 학습 실패, 셀 결과 신뢰 불가)')


In [ ]:
import subprocess, re, os, json, glob

log_path = os.path.join(RUN_DIR, 'train.log')
with open(log_path) as f:
    log = f.read()

for pat in [r'val-core/openai/gsm8k/[^\s]+', r'val-aux/num_turns/mean:np\.\w+\([0-9.]+\)',
            r'critic/score/mean:[0-9.]+', r'step:\d+.*']:
    matches = re.findall(pat, log)
    print(f'--- {pat} ({len(matches)} matches) ---')
    for m in matches[:15]:
        print(m[:300])
    print()

exit_ok = proc.returncode == 0
final_steps = [int(m) for m in re.findall(r'step:(\d+)', log)]
reached_100 = bool(final_steps) and max(final_steps) >= 100

strict_vals = [float(v) for v in re.findall(
    r'val-core/openai/gsm8k/(?:acc|reward)/mean@\d+:np\.\w+\(([0-9.]+)\)', log)]

print('=== 완주 확인 ===')
print(f'exit_code={proc.returncode}(0이어야 정상), 마지막 step={max(final_steps) if final_steps else None}, '
      f'val 포인트 수={len(strict_vals)}(11개여야 함: step0+10,20..100)')

print()
print('=== 판정기준 자동계산(사전 확정: step0 대비 +0.05 이상, 마지막 3포인트 비감소) ===')
if not exit_ok:
    print('학습이 비정상 종료됨(exit != 0) — 판정 보류. train.log 끝부분 직접 확인 필요.')
elif not reached_100 or len(strict_vals) < 11:
    print(f'100step 미완주 또는 val 포인트 부족(reached_100={reached_100}, '
          f'val개수={len(strict_vals)}) — 판정 보류.')
elif len(strict_vals) < 4:
    print(f'val 포인트 {len(strict_vals)}개 — 위 raw dump를 직접 확인, 정규식이 실제 key와 다를 수 있음.')
else:
    step0 = strict_vals[0]
    last3 = strict_vals[-3:]
    threshold = step0 + 0.05
    meets_threshold = all(v >= threshold for v in last3)
    non_decreasing = all(last3[i] <= last3[i+1] for i in range(len(last3)-1))
    verdict = "학습 진행" if (meets_threshold and non_decreasing) else "미달 — 옵션 1개만 변경(+100step 또는 lr 1e-6→3e-6)"
    print(f'step0={step0:.4f}, 임계값(step0+0.05)={threshold:.4f}')
    print(f'마지막 3개 val={last3}')
    print(f'임계값 이상: {meets_threshold}, 비감소: {non_decreasing}')
    print(f'판정: {verdict}')

# --- attribution 집계: 정답이 tool 출력에서 왔는지 assistant 텍스트에서 왔는지 ---
# validation_data_dir JSONL의 'output' 필드는 test_output_gen_batch.batch["responses"]를
# skip_special_tokens=True로 디코딩한 값(verl/trainer/ppo/ray_trainer.py:657-658,
# 소스 확인) — tool 메시지 포함 전체 응답 시퀀스. role 구분 토큰이 special token으로
# 분류돼 디코딩 시 사라질 가능성 있음(미검증) — 아래 tool_markers가 하나도 안 잡히면
# 그 가능성을 의심하고 raw jsonl 파일을 직접 열어 실제 구분자를 확인할 것.
print()
print('=== Attribution 집계 (정답 #### N 이 tool 출력발 vs assistant 자체발, 근사치) ===')
dump_files = sorted(glob.glob(os.path.join(val_dump_dir, '*.jsonl')))
if not dump_files:
    print(f'val_dump_dir({val_dump_dir})에 jsonl 없음 — validation_data_dir 동작 미검증, 직접 확인 필요.')
else:
    from_tool, from_assistant, from_neither, total = 0, 0, 0, 0
    for path in dump_files:
        with open(path) as f:
            for line in f:
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    continue
                output = rec.get('output', '')
                if not isinstance(output, str) or '####' not in output:
                    from_neither += 1
                    total += 1
                    continue
                total += 1
                last_hash = output.rfind('####')
                tool_markers = [m.start() for m in re.finditer(r'(?i)role.{0,10}tool|<\|tool', output)]
                if tool_markers and max(tool_markers) < last_hash:
                    from_tool += 1
                else:
                    from_assistant += 1
    print(f'val_dump 레코드 총 {total}개')
    if total:
        print(f'  #### 없음: {from_neither} ({from_neither/total:.1%})')
        print(f'  tool 출력 이후 #### (tool 귀속 추정): {from_tool} ({from_tool/total:.1%})')
        print(f'  tool 마커 없이 #### (assistant 귀속 추정, 마커가 실제로 안 잡혔을 위험 포함): '
              f'{from_assistant} ({from_assistant/total:.1%})')
        if from_tool == 0 and from_assistant > 0:
            print('  [주의] tool 마커가 하나도 안 잡힘 — role 구분자가 special token으로 '
                  '사라졌을 가능성. dump_files[0]의 첫 레코드를 직접 열어 raw output 텍스트 확인할 것.')

# env snapshot
freeze = subprocess.run(['bash', '-c',
    'cd /content/verl && uv run --frozen --all-packages --extra vllm --extra fsdp pip freeze'],
    capture_output=True, text=True).stdout
with open(os.path.join(RUN_DIR, 'pip_freeze.txt'), 'w') as f:
    f.write(freeze)

smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout
with open(os.path.join(RUN_DIR, 'nvidia-smi.txt'), 'w') as f:
    f.write(smi)

print()
print('Saved to', RUN_DIR)
print(subprocess.run(['ls', '-la', RUN_DIR], capture_output=True, text=True).stdout)


## 끝났으면

- `RUN_DIR` 아래 (`train.log`, `pip_freeze.txt`, `nvidia-smi.txt`, `val_dump/`)를 Drive에서 확인.
- control arm 결과(step0=0.08%→step100=48.0%)와 마지막 3개 val 비교.
- Attribution 집계에서 "tool 마커 없음" 경고가 뜨면 `val_dump/*.jsonl` 첫 레코드를
  직접 열어 raw output 텍스트 확인 — role 구분자가 사라졌을 가능성.
- `num_turns/max`가 학습 내내 2.0이었으면(조기중단 경고 무시하고 완주한 경우)
  tool이 사실상 안 쓰였다는 뜻 — val 결과가 control arm과 비슷해도 놀랄 일 아님.
- 이 세션은 종료해도 됨 — 결과는 Drive에 남아있음.
- 로컬 raw 결과는 `/mnt/c/.../Google Drive/agentic_rl/runs/...`(Google Drive Desktop 동기화 시).
- 완주 후 다음 착수점: 이 결과를 control arm과 비교해 STATE/JOURNAL에 반영 →
  컴파일러/레드팀 reward 설계로 전환(GSM8K 트랙 목표 달성).
